
# 04. Final Model & Kaggle Submission

## 1. Objective

This notebook trains the selected CatBoost model on the
complete labeled training dataset and generates predictions
for the Kaggle test dataset.

CatBoost was selected based on its superior mean
cross-validation MAE compared with XGBoost.

The final workflow includes:
- Preparing the full training and test datasets.
- Training the selected model on all labeled observations.
- Generating predictions for unseen test records.
- Creating a submission file in the required Kaggle format.


In [1]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

train = pd.read_csv(project_root / "data" / "raw" / "train.csv")
test = pd.read_csv(project_root / "data" / "raw" / "test.csv")

X = train.drop(columns=["id", "loss"])
y = train["loss"]

test_ids = test["id"]
X_test = test[X.columns]

cat_cols = X.select_dtypes(include=["object", "str", "category"]).columns.tolist()

print("Full training shape:", X.shape)
print("Test shape:", X_test.shape)
print("Categorical features:", len(cat_cols))

Full training shape: (188318, 130)
Test shape: (125546, 130)
Categorical features: 116



## 2. Final Model Training

CatBoost was selected based on its lower mean
cross-validation MAE compared with XGBoost.

The final model uses the selected 1,000-iteration
configuration and is trained on all 188,318 labeled
observations.

No validation split is used at this stage, allowing
the model to learn from the complete training dataset.


In [2]:
from catboost import CatBoostRegressor

final_model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function="MAE",
    random_seed=42,
    thread_count=2,
    allow_writing_files=False,
    verbose=200,
)

final_model.fit(
    X,
    y,
    cat_features=cat_cols,
)

0:	learn: 1773.9352876	total: 850ms	remaining: 14m 9s
200:	learn: 1176.1677559	total: 2m 40s	remaining: 10m 37s
400:	learn: 1148.1224281	total: 5m 37s	remaining: 8m 23s
600:	learn: 1133.5964220	total: 8m 39s	remaining: 5m 44s
800:	learn: 1124.2982029	total: 11m 42s	remaining: 2m 54s
999:	learn: 1118.4345634	total: 14m 39s	remaining: 0us


CatBoostRegressor(allow_writing_files=False, depth=6, iterations=1000, learning_rate=0.05, loss_function='MAE', random_seed=42, thread_count=2, verbose=200)

In [3]:
import numpy as np

test_pred = final_model.predict(X_test)

print(f"Predictions: {len(test_pred):,}")
print(f"Minimum prediction: {test_pred.min():,.2f}")
print(f"Maximum prediction: {test_pred.max():,.2f}")
print(f"Negative predictions: {(test_pred < 0).sum()}")
print(f"Non-finite predictions: {(~np.isfinite(test_pred)).sum()}")

Predictions: 125,546
Minimum prediction: -17.60
Maximum prediction: 23,509.36
Negative predictions: 1
Non-finite predictions: 0



### 3.1 Prediction Post-processing

The final model generated 125,546 predictions with no
missing or non-finite values.

One prediction was negative (-17.60). Since claim
severity cannot be negative, predictions are clipped
at zero to enforce a valid output range.

This adjustment affects only one test observation.


In [4]:
final_predictions = np.clip(test_pred, a_min=0, a_max=None)

print(f"Adjusted predictions: {(test_pred != final_predictions).sum()}")
print(f"Minimum prediction: {final_predictions.min():,.2f}")
print(f"Negative predictions: {(final_predictions < 0).sum()}")

Adjusted predictions: 1
Minimum prediction: 0.00
Negative predictions: 0



## 4. Model Persistence

The final CatBoost model is saved in its native binary
format for future inference.

Saving the trained model eliminates the need to repeat
the computationally expensive training process.

Prediction post-processing, including non-negative
clipping, remains a separate inference step.


In [5]:
model_dir = project_root / "models"
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "catboost_final.cbm"

final_model.save_model(str(model_path))

model_size_mb = model_path.stat().st_size / (1024**2)

print(f"Model saved to: {model_path}")
print(f"Model size: {model_size_mb:.2f} MB")

Model saved to: /Users/ebrietas/Allstate_Claims_Severity/models/catboost_final.cbm
Model size: 5.31 MB



### 4.1 Model Reload Verification

The saved CatBoost model is reloaded from disk and used
to generate predictions on the test dataset.

These predictions are compared with the original model
outputs to verify inference consistency.

Non-negative clipping is applied separately after
prediction.


In [6]:
reloaded_model = CatBoostRegressor()
reloaded_model.load_model(str(model_path))

reloaded_pred = reloaded_model.predict(X_test)

print("Predictions match:", np.allclose(test_pred, reloaded_pred))
print(
    "Maximum absolute difference:",
    np.max(np.abs(test_pred - reloaded_pred)),
)

reloaded_final_pred = np.clip(reloaded_pred, a_min=0, a_max=None)

print(f"Negative predictions after clipping: {(reloaded_final_pred < 0).sum()}")

Predictions match: True
Maximum absolute difference: 0.0
Negative predictions after clipping: 0



### 4.2 Model Persistence Findings

The final CatBoost model was successfully saved and
reloaded from its native `.cbm` format.

The reloaded model reproduced the original test
predictions exactly, with a maximum absolute
difference of **0.0**.

After applying non-negative clipping, all predictions
satisfied the valid claim severity range.

This confirms that the trained model can be reused
for inference without retraining.
